# Reminder: SCALE-I tools for DPA security analysis

As you have seen during SCALE-I, side-channel attacks are often split in two steps: 

1. The offline profiling steps, during which the adversary has access to a device similar to the one he aims to attack. During this step, he collects `q_t` (i.e., the training complexity) traces with known inputs in order to build a statistical model of the leakage traces conditioned to the sensitive intermediate states manipulated by the target circuitry. 
2. The online attack step, during which the adversary relies on the models he built in order to recover the sensitive data using up to `q_a` traces (i.e., the attack complexity).

The performances achieved by an attack directly depend on the quality of the model used, and SCALE-I training covered the topics of how one may evaluate attacks performances as well as informativness of statistical model following a worst-case evaluation strategy.  

As an introduction, the following cells cover the evaluations steps covered during the last part of SCALE-I. Your goal is to perform a quick security analysis of a software implementation of the AES-128, running on a STM32F415 (which will be used as the target circuitry during the whole training). The traces provided only cover the first round operations, running at 84MHz. 



## Point of Interest

Next, we rely on the SNR metric in order to identify the informative point in the traces (see Section 2.2.1 of the book for more details). This preliminary step is used to reduce the computing complexity of the training step, by modelling only the time samples that provide information while the other one are not considered. Besides that, it allows us to gather information about the target implementation. What do you observe? What does it tell you about the implementation? 

In [1]:
# Import the useful module for the session
from utils_scale import utils_files, utils_aes, utils_plot, utils_IT, utils_ta
import numpy as np
from scalib.metrics import SNR

In [2]:
# Load the dataset
ds = utils_files.load_dataset(utils_files.DS_CFG['sw-aes_training'], seed_shuffle=0, remove_first=True)

# Amount of traces provided
am_traces = ds["traces"].shape[0]
print(f"There are {am_traces} traces in the dataset")

In [3]:
# Compute the intermediate variable, which is the output of the Sbox.
labels = utils_aes.Sbox[ds['pts'] ^ ds['ks']]

# Compute the SNR associated to each classes with SCAlib
snrobj = SNR(nc=256)
snrobj.fit_u(
    ds['traces'].astype(np.int16),
    labels.astype(np.uint16)
)
snrs = snrobj.get_snr()


In [5]:
# Display the SNR and examplary traces (with the average trace in dashed red)
utils_plot.display_snr_SBout(snrs, ds['traces'][:10], use_log_scale=False)

## Training phase and model quality assesment

The next steps consists in creating the statistical model. During SCALE-I, we relied on multivariate gaussian template together with reduction of dimensionality using LDA in order to identify a linear subspace of `p` dimensions (which is equal or smaller than the total amount of POI considered). In this context, two main parameters of the profilling phase are the amount of POIs to used, and the value of `p`. See Section 2.3.2 of the book or [SCALib documentation](https://scalib.readthedocs.io/en/stable/source/api/scalib.modeling.LdaAcc.html) for more details.

The first steps is to identify the proper set of parameters, i.e., the value that lead to the most informative model. As a reminder, computing the exact Mutual Information (MI) between the traces and our model is infeasible in practice since it would require to know the (unknown) exact probability distribution of the leakage (which is exactly the purpose of the purpose of the modelling phase). As a proxy, we rely on the Perceived Information, a lower bound to the MI which quantifies the amount of information that can be extracted from a given model. When used together with the Theoritical Information (TI), an upper bound to the PI, an evaluator is able the assess the amount of information that can be exploited using his model, and evaluated if better performance would be achievable if a bigger training complexity was available. See Section 3.3.1 of the book for more details. 

In order to perform the evaluation, the following cells allow you to compute exhaustively the useful IT metrics values associated to Multivariate Gaussian models with arbitrary sets of parameters. In this configuration and using the available dataset, what is the best information level about each byte that you can confidently recover per trace? How would you describe the training complexity available (e.g., too low) and what is the minimal training complexity you would recommend in this setting? 


In [5]:
# Configuration of the parameters sets to explore. 

# Parameters selections to explore
explo_npois = [1] # Amount of POIs to use.
explo_ndims = [1] # Amount of dimension to use for the linear subspace.
q_t = 2000 # Training complexity to use. 


# Compute the intermediate variable, which is the output of the Sbox.
# (Same as the labels in the SNR above)
labels = utils_aes.Sbox[ds['pts'] ^ ds['ks']]

explos_params = utils_IT.explore_params_LDA(ds['traces'], labels, 2048, explo_npois, explo_ndims, q_t=q_t, chunk_size=10000)


In [6]:
# Display the heatmap associated to the parameters exploration.
# For each byte (ordered per column, from top left to bottom right), each cell display the IT metric obtained for a parameter set. 
# In the boxes bottom half are displayed the PI value obtained, while the top half displays the TI. 
utils_plot.make_heatmap(explos_params)

In [7]:
# Here, you choose the parameters sets to use during the training phase for each byte
# (i.e., amount of POIs and amount of dimension in the linear subspace.
# The following config is the default one, but tuning it a bit may be a good idea. 
# Caution: you have to provide the param set for all the bytes, in order (i.e., be sur that the 16 config are there ;) ). 
param_set = [
    (0, 1, 1), # Byte-0, using 1 POIs and 1 dimensions 
    (1, 2, 1), # Byte-1, using 2 POIs and 1 dimensions 
    (2, 2, 2), # Byte-2, using 2 POIs and 2 dimensions 
    (3, 3, 2), # Byte-3, using 3 POIs and 2 dimensions 
    (4, 1, 1), # ...
    (5, 1, 1),
    (6, 1, 1), 
    (7, 1, 1), 
    (8, 1, 1), 
    (9, 1, 1),
    (10, 1, 1),
    (11, 1, 1), 
    (12, 1, 1), 
    (13, 1, 1), 
    (14, 1, 1), 
    (15, 1, 1) 
]



In [8]:
# This cell allows you to compute the PI/TI curves as a function of the training complexity, for the parameter set
# of your choice. Besides the parameters, you can choose the training complexities you whish to compute the 
# TI and the PI for, as well a the amount of traces to use to compute the later. 

# Compute the PI curve for the 16 bytes
qt_s = [2048, 4096] # Training complexities to test
ntraces_testing = 10 # Amount of indep. test traces used to estimate the PI with the trained model. 



pis_curves = utils_IT.compute_PI_curves(
    ds['traces'][ntraces_testing:,:],
    labels[ntraces_testing:,:],
    ds['traces'][:ntraces_testing,:],
    labels[:ntraces_testing,:],
    qt_s,
    param_set,
    chunk_size=10000
)

tis_curves = utils_IT.compute_TI_curves(
    ds['traces'][ntraces_testing:,:],
    labels[ntraces_testing:,:],
    qt_s,
    param_set,
    chunk_size=10000
)


In [9]:
# This cell allows you to display the curves with the results computed in the previous cells. 
# For each bytes, you'll see the PI depicted in solid line, the TI in dashed line, and the colored area is the 
# confidence interval of the PI. 

# Choose the byte for which you want to display the results
byte_indexes = range(16) # Put any list of index here if you want

# Display the IT curves for the bytes indexes that you want
utils_plot.display_IT_results(
    byte_indexes,
    [
        ("LDA", pis_curves),
        ("LDA", tis_curves)
    ],
    scale=0.6,
    disable_legend=True
)


## Attack and Key rank estimation

Complementarily, the following cells perform template attacks against 5 different validation datsets and commpute the median key ranks achieved by the models. As a reminder, the key rank is a metric that allows to evaluate the performance of a practical attack (knowing the secret key to recover), by quantifying the position of the correct key in the list of the most probable (i.e., rank 1) to the least probable  one computed from the probabilities returned by the model. See Section 3.3.2 of th book for more details. 

Considering that an enumeration power of $2^{32}$ is available, what is the best attack complexity (i.e., the lowest) you achieve to recover the key in full (i.e, the 128-bits)? How does the situation evolve for $2^{64}$? Are these results coherent with you previous analysis?

In [10]:
# Perform the TA for several attack complexities
q_as = [512, 1024] # Attack complexities to perform the attack for
q_t = 2000 # Training complexity used


# Perform the attacks
atcks_res = utils_ta.explore_TA_multivariate_best(
    utils_files.DS_CFG['sw-aes_training'],
    utils_files.DS_CFG['sw-aes_atcks'],
    q_t,
    q_as,
    param_set,
)

In [11]:
# Bytes to consider for ranks computation.
# Using the 16 bytes indexes corresponds to computing the rank for the full 128-bit key.
# Using a samples of range(16) corresponds to ocmputing the rank for the subpart of the key only.
key_bytes_ranks = range(16)

utils_plot.display_ranks_full_key([atcks_res], key_bytes_ranks)

# Introduction to leakage resilience: SPA security and the LR-PRF case.

DPA attack turn out to be powerful in practice, as observed in the previous section. While the attacks performances depend on the target circuitry and the computational power of the adversary, an observation is that these become more difficult to perform in a SPA setting, that is when only the leakage associated to a limited amount of different plaintexts can be observed (see Section 3.1.2 of the book for more details). 

This observation led to the design of  primitives that limit the security threat posed by DPA, by limiting the number of plaintexts for which the adversary can combine the measurements (e.g., thanks to re-keying). A popular example is a tree based LR-PRF, which has been shown to achieve good properties for improving security against leakage. Its working principle[¹] is to process a 128-bit input (denoted $x$) sequentially per smaller words of $1 \leq n_x \leq 8$ bits, where each word is used to select among $2^{n_x}$ fixed different plaintext one that is encrypted in order to generate the key used at the following stage (and using the 128-bit key as the first stage key). As a result, the performances of the PRF is closely linked to $n_x$, since an execution of the LR-PRF requires $128/n_{x}$ AES-128 encryptions. 

Since a picture is sometimes worth a thousand words, the following figure represents the instanciation using $n_x = 2$. At each stage, the processed 2-bit word is used to select the plaintext among [$p_0$, $p_1$, $p_2$, $p_3$] and the resulting ciphertext is used as the key of the following stage. 

<div style="text-align:center"><img src="figures/lrprf.png" /></div>

In the following, we study the security of different implementation of this LR-PRF, so be sure to understand the working principle of the latter before continuing. In particular, what would be an attack path to follow in order to recover the value of the key $k$?

As a first implementation, we propose to rely on the SW AES implementation evaluated in the previous Section. Do you think that the LR-PRF would be secure for any value of $n_x$? If yes, why? If no, what would be the maximal acceptable value considering adversarial key enumeration capabilities of $2^{32}$ and $2^{64}$? 

[¹]: The original proposal describes a general form using a block cipher. We rely on the AES-128 here. 

## LR-PRF: HW AES variant

Alternatively, we propose to use a HW implementation of the AES instead of the SW one we used previously. To do so, we can leverage the AES coprocessor embedded on the STM32F415 we are measuring. The next cells re-implement the evaluation steps for traces measured from the AES co-processor (covering the full execution). How does this solution compare to the SW one when integrated in the LR-PRF? Would you say that the LR-PRF would be secure for any value of $n_{x}$? If yes, why? If no, what would be the maximal acceptable value, considering adversarial key enumeration capabilities of $2^{32}$ and $2^{64}$? How do you explain these differences?

*Remark: To make things easier for you, the following cells automatically detect the best sets of parameters based on the exploration results. However, this does not prevent you from entering them manually as before if you wish to test a different configuration!*

In [12]:
# Load the dataset
ds = utils_files.load_dataset(utils_files.DS_CFG['hw-aes_training'], seed_shuffle=0, remove_first=True)

# Compute the intermediate variable, which is the output of the Sbox.
labels = utils_aes.Sbox[ds['pts'] ^ ds['ks']]

# Compute the SNR associated to each classes with SCAlib
snrobj = SNR(nc=256)
snrobj.fit_u(
    ds['traces'].astype(np.int16),
    labels.astype(np.uint16)
)
snrs = snrobj.get_snr()
# Display the SNR
utils_plot.display_snr_SBout(snrs, ds['traces'][:10])

In [13]:
# Configuration of the parameters sets to explore
explo_npois = [1]
explo_ndims = [1]
q_t = 2500 # Use None to use all available


# Compute the intermediate variable, which is the output of the Sbox.
# (Same as the labels in the SNR above)
labels = utils_aes.Sbox[ds['pts'] ^ ds['ks']]

explos_params = utils_IT.explore_params_LDA(ds['traces'], labels, 2048, explo_npois, explo_ndims, q_t=q_t, chunk_size=10000)

# Identify the best param set from the exploration results
best_param_sets = utils_IT.identify_best_params(explos_params)

In [14]:
# Display the resulting heatmaps.
utils_plot.make_heatmap(explos_params)

In [15]:
# Compute the PI curve for the 16 bytes
qt_s = [2500, 4096] # Training complexities to test
ntraces_testing = 10 # Amount of indep. test traces used to estimate the PI with the trained model. 


pis_curves = utils_IT.compute_PI_curves(
    ds['traces'][ntraces_testing:,:],
    labels[ntraces_testing:,:],
    ds['traces'][:ntraces_testing,:],
    labels[:ntraces_testing,:],
    qt_s,
    best_param_sets,
    chunk_size=10000
)

tis_curves = utils_IT.compute_TI_curves(
    ds['traces'][ntraces_testing:,:],
    labels[ntraces_testing:,:],
    qt_s,
    best_param_sets,
    chunk_size=10000
)


In [16]:
# Choose the byte for which you want to display the results
byte_indexes = range(16) # Put any list of index here if you want

# Display the IT curves for the bytes indexes that you want
utils_plot.display_IT_results(
    byte_indexes,
    [
        ("LDA", pis_curves),
        ("LDA", tis_curves)
    ],
    scale=0.6,
    disable_legend=True
)


In [17]:
q_as = [512, 1024] # Attack complexities to perform the attack for
q_t = 2500 # Training complexity used


param_set = best_param_sets

# Perform the attacks
atcks_res = utils_ta.explore_TA_multivariate_best(
    utils_files.DS_CFG['hw-aes_training'],
    utils_files.DS_CFG['hw-aes_atcks'],
    q_t,
    q_as,
    param_set,
    chunk_size=5000
)

In [18]:
# Bytes to consider for ranks computation.
# Using the 16 bytes indexes corresponds to computing the rank for the full 128-bit key.
# Using a samples of range(16) corresponds to ocmputing the rank for the subpart of the key only.
key_bytes_ranks = range(16)

utils_plot.display_ranks_full_key([atcks_res], key_bytes_ranks)

## LR-PRF: HW AES variant with repSPA

Despite limiting by the design the amount of different plaintexts available, the LR-PRF does not avoid to repeat similar plaintexts several times. This may be exploited to reduce the physical noise leveraging the averaging of multiple measurements into less noisy traces (see Section 4.1.2 of the book for more details).

Once again, your goal is to evaluate the security of the LR-PRF with an HW implementation of the AES, but in the avgSPA adversarial setting. To do so, we collected traces that where averaged 10 and 100 times, for which the evaluations steps are implemented in the following cells as previously. How does the security evolve? Could we do better[¹]?

[¹]: *hint: The Section 6 of [Primitive-Level vs. Implementation-Level DPASecurity: a Certified Case Study(Pleading for Standardized Leakage-Resilient Cryptography)](https://tches.iacr.org/index.php/TCHES/article/view/12234/12045) might provide some information.*

In [19]:
# Load the dataset
# Use either "hw-aes-avg10_training" for x10, or "hw-aes-avg100_training" for x100
ds = utils_files.load_dataset(utils_files.DS_CFG["hw-aes-avg10_training"], seed_shuffle=0, remove_first=True)

# Compute the intermediate variable, which is the output of the Sbox.
labels = utils_aes.Sbox[ds['pts'] ^ ds['ks']]

# Compute the SNR associated to each classes with SCAlib
snrobj = SNR(nc=256)
snrobj.fit_u(
    ds['traces'].astype(np.int16),
    labels.astype(np.uint16)
)
snrs = snrobj.get_snr()
# Display the SNR
utils_plot.display_snr_SBout(snrs, ds['traces'][:10])

In [20]:
# Change here
explo_npois = [1]
explo_ndims = [1]
q_t = 2500 # Use None to use all available


# Compute the intermediate variable, which is the output of the Sbox.
# (Same as the labels in the SNR above)
labels = utils_aes.Sbox[ds['pts'] ^ ds['ks']]

explos_params = utils_IT.explore_params_LDA(ds['traces'], labels, 2048, explo_npois, explo_ndims, q_t=q_t, chunk_size=10000)

# Identify the best param set from the exploration results
best_param_sets = utils_IT.identify_best_params(explos_params)

In [21]:
# Display the resulting heatmaps.
utils_plot.make_heatmap(explos_params)

In [22]:
# Compute the PI curve for the 16 bytes
qt_s = [2500, 4096] # Training complexities to test
ntraces_testing = 10 # Amount of indep. test traces used to estimate the PI with the trained model. 

### ANSWER_START
qt_s = [32768, 65536, 131072, 262144, 500000] # Training complexities to test
ntraces_testing = 2048 # Amount of indep. test traces used to estimate the PI with the trained model. 
### ANSWER_STOP

pis_curves = utils_IT.compute_PI_curves(
    ds['traces'][ntraces_testing:,:],
    labels[ntraces_testing:,:],
    ds['traces'][:ntraces_testing,:],
    labels[:ntraces_testing,:],
    qt_s,
    best_param_sets,
    chunk_size=10000
)

tis_curves = utils_IT.compute_TI_curves(
    ds['traces'][ntraces_testing:,:],
    labels[ntraces_testing:,:],
    qt_s,
    best_param_sets,
    chunk_size=10000
)


In [23]:
# Choose the byte for which you want to display the results
byte_indexes = range(16) # Put any list of index here if you want

# Display the IT curves for the bytes indexes that you want
utils_plot.display_IT_results(
    byte_indexes,
    [
        ("LDA", pis_curves),
        ("LDA", tis_curves)
    ],
    scale=0.6,
    disable_legend=True
)


In [24]:
# Perform the TA for several attack complexities
q_as = [512, 1024] # Attack complexities to perform the attack for
q_t = 2500 # Training complexity used


param_set = best_param_sets

# Perform the attacks
atcks_res = utils_ta.explore_TA_multivariate_best(
    utils_files.DS_CFG["hw-aes-avg100_training"],
    utils_files.DS_CFG["hw-aes-avg100_atcks"],
    q_t,
    q_as,
    param_set,
    chunk_size=10000
)

In [25]:
# Bytes to consider for ranks computation.
# Using the 16 bytes indexes corresponds to computing the rank for the full 128-bit key.
# Using a samples of range(16) corresponds to ocmputing the rank for the subpart of the key only.
key_bytes_ranks = range(16)

utils_plot.display_ranks_full_key([atcks_res], key_bytes_ranks)